# [SK 01 - Chat Completion with GetChatMessageContent (no kernel, no plugins)](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python#writing-your-first-console-app)
With just a few lines of code, we show how to use a chat completion model to answer user's questions that the model answers without leveragin any external tools.<br/>
## Environment configuration
```
conda create -n semantic_kernel python=3.13 -y
conda activate semantic_kernel
pip install semantic-kernel python-dotenv jupyter
jupyter kernelspec uninstall semantic_kernel -y
python -m ipykernel install --name semantic_kernel --user
```

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

Environment variables have been loaded ;-)
os.environ['AZURE_OPENAI_ENDPOINT']: https://aiservicesiyva.openai.azure.com/


# Create an `AzureChatCompletion` object e.g. the `assistant` from SK library

In [2]:
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

azure_chat_completion=AzureChatCompletion(service_id="default")
azure_chat_completion

AzureChatCompletion(ai_model_id='gpt-4.1', service_id='default', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x7de1aef746e0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)

# Create a user message and add it to a blank history

In [3]:
from semantic_kernel.contents.chat_history import ChatHistory

# Create a blank history of the conversation
history = ChatHistory() # initially blank

# Add user input to the history
history.add_user_message("Tell me what is Azure in less than 10 words.")

history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Tell me what is Azure in less than 10 words.', encoding=None)], encoding=None, finish_reason=None, status=None)])

# Define settings for the chat
For the moment we're just creating the `AzureChatPromptExecutionSettings` object without any specific configuration.<br/>
For example, `function_choice_behavior=None` meaning that the kernel is not asked to identify the functions according to the function call paradigm.<br/>
This means that we don't even need a kernel instance when we call `get_chat_message_contents` in the next code cell.

In [4]:
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import AzureChatPromptExecutionSettings

# Create default settings for the chat conversation
execution_settings = AzureChatPromptExecutionSettings()
execution_settings

AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=None, ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, parallel_tool_calls=None, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, max_completion_tokens=None, reasoning_effort=None, extra_body=None)

# Put all together
- Azure Chat Completion (the OpenAI Assistant from SK Library)
- History (including the first user message)
- Settings (empty, in this case)

The `result` we get is a list of semantic_kernel.contents.chat_message_content.`ChatMessageContent` objects, whose `content` field contains the message text.

In [5]:
result = await azure_chat_completion.get_chat_message_contents(
    chat_history=history,
    settings = execution_settings,
    # kernel is NOT needed here because we have no tools, 
    # nor we configured the kernel through execution_settings to use them
)

# Add result to the history
for r in result:
    history.add_message(r)

#  Print the results
i=0
for m in history.messages:
    i += 1
    print(f"\nMessage {i}:")
    display(m)

print(f"\nFinal answer: {history.messages[-1].inner_content.choices[0].message.content}")


Message 1:


ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Tell me what is Azure in less than 10 words.', encoding=None)], encoding=None, finish_reason=None, status=None)


Message 2:


ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-Be2RRfYqw2IXeu99yD2tIcHhjGtDY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Microsoft's cloud platform for computing, storage, and services.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'filtered': False, 'detected': False}, 'protected_material_text': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1748882205, model='gpt-4.1-2025-04-14', object='chat.completion', service_tier=None, system_fingerprint='fp_07e970ab25', usage=CompletionUsage(completion_tokens=13, prompt_tokens=19, total_tokens=32, completion_tokens_details=CompletionTokensDetails(accepted_predic


Final answer: Microsoft's cloud platform for computing, storage, and services.
